# # YOLO Annotatoions Generator
This notebook is used to generate YOLO annotations by using a pre-trained YOLO model.

## Imports

In [ ]:
import sys
import cv2
from pathlib import Path
import json
import shutil

In [ ]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

In [ ]:
from src.utils.config.config import Config
from src.detector.yolo_detector import YOLODetector

## Config

In [ ]:
config_loader = Config()
cfg = config_loader.load_config()

yolo_model_type = cfg.paths.models / config_loader.get("model", "yolo.pretrained")
yolo_imgsz = config_loader.get("model", "yolo.imgsz")

raw_dataset = cfg.project_root / config_loader.get("paths", "dataset.raw")
raw_dataset = raw_dataset / "CafeV1"

output_dataset = cfg.project_root / config_loader.get("paths", "dataset.processed") / "CafeV1" 

In [ ]:
ALLOWED_CLASSES = [
    0,   # person
    63,  # laptop
    67,  # cell phone
    73   # book
]

## Init YOLO

In [ ]:
detector = YOLODetector(yolo_model_type, cfg.project_root)

## Functions

### load_existing_json

In [ ]:
def load_existing_json(json_path):
    if json_path.exists():
        with open(json_path, "r") as f:
            return json.load(f)
    # Empty data
    return None

### save_json

In [ ]:
def save_json(data, json_path):
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)

### process_image

In [ ]:
def process_image(image_path):
    # Do inference on the frame
    results = detector.model(
        source=image_path,
        verbose=False,
        imgsz = yolo_imgsz, # 640, 960, 1280,
        classes = ALLOWED_CLASSES,
        conf=0.15
    )[0]

    # YOLO built-in visualizer (used to verify)
    # results.show()

    detections = []
    if results.boxes is None:
        return detections

    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    for bbox, cls in zip(boxes, classes):
        cls = int(cls)
        x1, y1, x2, y2 = bbox.tolist() 
         
        # Format into JSON
        detections.append({
            "bbox": [x1, y1, x2, y2],
            "class_id": cls
        })

    return detections

### process_clip

In [ ]:
def process_clip(clip_path):
    clip_name = clip_path.name # 0
    
    images_path = clip_path / "images" 
    
    output_path = output_dataset / "clips" / clip_name
    output_images_path = output_path / "images"  
    output_json_path = output_path / "anns.json"
    
    if not images_path.exists():
        print(f"Couldn't process clip {clip_name}")
        return

    # Get and sort frames
    image_files = (p for p in images_path.iterdir()) # list of frames
    image_files = sorted(image_files, key=lambda x: int(x.stem.split("_")[1]))

    # Ensure path exists
    output_images_path.mkdir(parents=True, exist_ok=True)

    json_data = load_existing_json(output_json_path)
    
    # Create default JSON if not found
    if json_data == None:
        json_data = {
            "clip_id": clip_name,
            "frames": []
        }
    
    processed_frames = {
        f["frame_id"] 
        for f in (json_data.get("frames") or [])
    }

    for frame_id, image_file in enumerate(image_files):
        # Skip already processed frames
        if frame_id in processed_frames:
            continue 

        image_path = images_path / image_file

        detections = process_image(image_path)

        # Add detections data for current frame
        json_data["frames"].append({
            "frame_id": frame_id,
            "objects": detections
        })
        
        # Save frame image
        shutil.copy2(image_path, output_images_path)
        # break # temp - used to validate one frame

    save_json(json_data, output_json_path)
    # print(f"Saved anns for clip {clip_name} at {output_json_path}")

## Run

In [ ]:
def main():
    clips_path = raw_dataset / "Clips"
    clip_dirs = sorted(p for p in clips_path.iterdir() if p.is_dir())

    for clip_path in clip_dirs:
        process_clip(clip_path)
        # return # temp - used to validate one clip

main()